# Bab 2 — K-Means Clustering

Notebook pendamping Bab 2 Level 4.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

## Learning Objectives

- Menjelaskan dua langkah inti K-Means: *assignment* dan *update*.
- Memahami fungsi objektif WCSS dan konvergensi.
- Membedakan inisialisasi `random` dan `k-means++`, dan fungsi `n_init`.
- Menulis K-Means dari nol dengan NumPy dan membandingkannya dengan scikit-learn.
- Menerapkan K-Means pada Mall Customers dan melabeli pelanggan baru.

## Konsep singkat

K-Means membagi data menjadi K kelompok. Dua langkah diulang sampai konvergen: (1) tiap titik masuk ke centroid terdekat, (2) centroid baru = rata-rata anggota kelompok. Fungsi objektifnya WCSS: jumlah kuadrat jarak titik ke centroid kelompoknya.

## Dataset

8 titik buatan tangan (hitungan manual), data `make_blobs` 4 pusat, dan Mall Customers (200 pelanggan). `random_state=42` dipakai di mana-mana.

## Langkah 0: Pindah ke folder utama proyek

In [ ]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("../..")

## Bagian A: Siapkan alat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

## Bagian B: Data kecil untuk ditelusuri

Centroid awal sengaja dipilih buruk: keduanya di gerombol kiri.

In [ ]:
# Delapan titik di dua gerombol: kiri bawah dan kanan atas
X_tiny = np.array(
    [
        [1, 1], [2, 1], [1, 2], [2, 2],   # gerombol kiri bawah
        [6, 5], [7, 5], [6, 6], [7, 6],   # gerombol kanan atas
    ],
    dtype=float,
)
# Centroid awal sengaja dipilih buruk: dua-duanya di gerombol kiri
initial_centroids = np.array([[1.0, 1.0], [2.0, 1.0]])
print("Bentuk X_tiny:", X_tiny.shape)
print("Centroid awal:\n", initial_centroids)

## Bagian C: K-Means dari nol

Assignment memakai broadcasting NumPy dan `argmin`. Update memakai rata-rata anggota. Kelompok kosong mempertahankan centroid lama.

In [ ]:
def kmeans_scratch(X, initial_centroids, max_iter=100, tol=1e-6):
    """K-Means sederhana. Mengembalikan label, centroid akhir, dan riwayat tiap iterasi."""
    centroids = initial_centroids.copy()
    k = len(centroids)
    history = []

    for iteration in range(1, max_iter + 1):
        # Langkah 1 (assignment): jarak setiap titik ke setiap centroid, bentuk (n, k)
        distances = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        labels = distances.argmin(axis=1)

        # WCSS: jumlah kuadrat jarak titik ke centroid kelompoknya (centroid sebelum diperbarui)
        wcss = ((X - centroids[labels]) ** 2).sum()

        # Langkah 2 (update): centroid baru = rata-rata anggota. Kelompok kosong: centroid tetap
        new_centroids = np.array(
            [
                X[labels == j].mean(axis=0) if np.any(labels == j) else centroids[j]
                for j in range(k)
            ]
        )

        # Seberapa jauh centroid bergeser pada iterasi ini
        shift = np.linalg.norm(new_centroids - centroids, axis=1).max()
        history.append(
            {"iteration": iteration, "labels": labels, "centroids": centroids.copy(),
             "wcss": wcss, "shift": shift}
        )
        centroids = new_centroids

        if shift <= tol:  # centroid tidak bergerak lagi: berhenti
            break

    return labels, centroids, history

## Bagian D: Menelusuri 8 titik

Cocokkan dengan hitungan manual di halaman bab: WCSS 166 → 21,111 → 4.

In [ ]:
labels_tiny, centroids_tiny, hist_tiny = kmeans_scratch(X_tiny, initial_centroids)

for step in hist_tiny:
    print(f"Iterasi {step['iteration']}:")
    print("  centroid yang dipakai :", step["centroids"].round(3).tolist())
    print("  label tiap titik      :", step["labels"].tolist())
    print(f"  WCSS                  : {step['wcss']:.3f}")
    print(f"  pergeseran centroid   : {step['shift']:.3f}")

print("\nCentroid akhir:", centroids_tiny.round(3).tolist())

In [ ]:
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]

fig, axes = plt.subplots(1, len(hist_tiny), figsize=(4 * len(hist_tiny), 4), sharex=True, sharey=True)
for ax, step in zip(axes, hist_tiny):
    for j in range(2):
        pts = X_tiny[step["labels"] == j]
        ax.scatter(pts[:, 0], pts[:, 1], s=60, color=palette[j])
        cx, cy = step["centroids"][j]
        ax.scatter(cx, cy, s=200, marker="X", color=palette[j], edgecolor="black")
    ax.set_title(f"Iterasi {step['iteration']} (WCSS = {step['wcss']:.2f})")
    ax.set_xlabel("x1")
axes[0].set_ylabel("x2")
plt.tight_layout()
plt.show()

## Bagian E: K-Means scikit-learn pada data blobs

In [ ]:
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.9, random_state=42)

kmeans = KMeans(n_clusters=4, init="k-means++", n_init=10, max_iter=300, tol=1e-4, random_state=42)
labels_sk = kmeans.fit_predict(X)

print("Ukuran kelompok:", np.bincount(labels_sk).tolist())
print("Centroid (cluster_centers_):\n", kmeans.cluster_centers_.round(3))
print("Inertia (WCSS):", round(kmeans.inertia_, 3))
print("Jumlah iterasi (n_iter_):", kmeans.n_iter_)

## Bagian F: Versi buatan sendiri vs scikit-learn

In [ ]:
# Jalankan versi buatan sendiri dari centroid awal yang kita pilih sendiri
rng = np.random.default_rng(42)
start = X[rng.choice(len(X), size=4, replace=False)]
labels_me, centroids_me, hist_me = kmeans_scratch(X, start)

# Urutkan centroid supaya nomor kelompok tidak mengganggu perbandingan
sorted_me = centroids_me[np.lexsort((centroids_me[:, 1], centroids_me[:, 0]))]
sorted_sk = kmeans.cluster_centers_[np.lexsort((kmeans.cluster_centers_[:, 1], kmeans.cluster_centers_[:, 0]))]

print("Iterasi versi buatan sendiri:", len(hist_me))
print("WCSS versi buatan sendiri   :", round(hist_me[-1]["wcss"], 3))
print("Selisih terbesar antar centroid:", np.abs(sorted_me - sorted_sk).max().round(6))

## Bagian G: Pengaruh K dan elbow

WCSS selalu turun saat K naik. Kita mencari titik tempat penurunan melandai, bukan WCSS terkecil.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
for ax, k in zip(axes, [2, 3, 4, 6]):
    model = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    for j in range(k):
        pts = X[model.labels_ == j]
        ax.scatter(pts[:, 0], pts[:, 1], s=14, color=palette[j % len(palette)])
    ax.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1],
               s=160, marker="X", color="black")
    ax.set_title(f"K = {k}  (WCSS = {model.inertia_:.0f})")
    ax.set_xlabel("x1")
axes[0].set_ylabel("x2")
plt.tight_layout()
plt.show()

In [ ]:
k_values = range(1, 9)
wcss_values = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).inertia_ for k in k_values]
for k, w in zip(k_values, wcss_values):
    print(f"K = {k}: WCSS = {w:.1f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(list(k_values), wcss_values, marker="o", color="#0072B2")
ax.set_xlabel("Jumlah kelompok K")
ax.set_ylabel("WCSS (inertia)")
ax.set_title("WCSS terhadap K: penurunan tajam berhenti di K = 4")
plt.tight_layout()
plt.show()

## Bagian H: Inisialisasi random vs k-means++

`n_init=1` dipakai agar pengaruh titik awal terlihat apa adanya.

In [ ]:
# Data yang lebih menantang: lima gerombol yang cukup berdekatan
X5, _ = make_blobs(n_samples=500, centers=5, cluster_std=1.1, random_state=3)

rows = []
for init in ["random", "k-means++"]:
    inertias, iters = [], []
    for seed in range(20):
        model = KMeans(n_clusters=5, init=init, n_init=1, random_state=seed).fit(X5)
        inertias.append(model.inertia_)
        iters.append(model.n_iter_)
    rows.append(
        {"init": init, "WCSS terkecil": min(inertias), "WCSS terbesar": max(inertias),
         "rata-rata iterasi": np.mean(iters),
         "jumlah seed dengan WCSS terbaik": int(np.sum(np.isclose(inertias, min(inertias), rtol=1e-6)))}
    )
print(pd.DataFrame(rows).round(2).to_string(index=False))

In [ ]:
def kmeans_plus_plus_init(X, k, seed=0):
    """Memilih k centroid awal dengan k-means++ (versi sederhana)."""
    rng = np.random.default_rng(seed)
    centers = [X[rng.integers(len(X))]]  # centroid pertama: satu titik acak
    for _ in range(k - 1):
        # D(x)^2: kuadrat jarak tiap titik ke centroid terdekat yang sudah dipilih
        d2 = np.min([((X - c) ** 2).sum(axis=1) for c in centers], axis=0)
        probs = d2 / d2.sum()  # makin jauh, makin besar peluang terpilih
        centers.append(X[rng.choice(len(X), p=probs)])
    return np.array(centers)

centers_pp = kmeans_plus_plus_init(X5, k=5, seed=0)
print("Centroid awal dari k-means++ (buatan sendiri):\n", centers_pp.round(2))
_, _, hist_pp = kmeans_scratch(X5, centers_pp)
print("Iterasi sampai konvergen:", len(hist_pp), "| WCSS akhir:", round(hist_pp[-1]["wcss"], 2))

## Bagian I: Konvergensi

Tidak ada peringatan bila `max_iter` terlalu kecil, jadi periksa `n_iter_` sendiri.

In [ ]:
for max_iter in [1, 2, 3, 300]:
    model = KMeans(n_clusters=4, init="random", n_init=1, max_iter=max_iter, random_state=0).fit(X)
    print(f"max_iter = {max_iter:<3} -> n_iter_ = {model.n_iter_}, WCSS = {model.inertia_:.2f}")

wcss_history = [float(step["wcss"]) for step in hist_me]
print("\nWCSS tiap iterasi (versi buatan sendiri):", [round(w, 1) for w in wcss_history])
print("Tidak pernah naik?", all(a >= b - 1e-9 for a, b in zip(wcss_history, wcss_history[1:])))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([s["iteration"] for s in hist_me], wcss_history, marker="o", color="#0072B2")
ax.set_xlabel("Iterasi")
ax.set_ylabel("WCSS")
ax.set_title("WCSS turun lalu mendatar saat K-Means konvergen")
plt.tight_layout()
plt.show()

## Bagian J: K-Means pada Mall Customers

Fitur diskalakan dulu. Centroid dikembalikan ke satuan asli dengan `inverse_transform`.

In [ ]:
df = pd.read_csv("data/mall_customers.csv")
features = ["Annual Income (k$)", "Spending Score (1-100)"]

scaler = StandardScaler()
X_mall = scaler.fit_transform(df[features])  # rata-rata 0, simpangan baku 1

km_mall = KMeans(n_clusters=5, n_init=10, random_state=42)
df["Cluster"] = km_mall.fit_predict(X_mall)

print("Jumlah pelanggan per kelompok:", np.bincount(df["Cluster"]).tolist())
# Kembalikan centroid ke satuan asli supaya mudah dibaca
centers_original = scaler.inverse_transform(km_mall.cluster_centers_)
print(pd.DataFrame(centers_original, columns=features).round(1))

### Memberi label pada pelanggan baru

Pakai `transform` dengan scaler yang sama, bukan `fit_transform`.

In [ ]:
new_customers = pd.DataFrame([[30, 80], [90, 15]], columns=features)
new_scaled = scaler.transform(new_customers)  # pakai scaler yang SAMA, jangan fit ulang
print("Label pelanggan baru:", km_mall.predict(new_scaled).tolist())
print("Jarak ke tiap centroid (pelanggan pertama):", km_mall.transform(new_scaled)[0].round(2).tolist())

### Kesalahan umum: lupa menyamakan skala

In [ ]:
# Kesalahan umum: satu fitur berskala jauh lebih besar (pendapatan dalam dolar, bukan ribu dolar)
X_unscaled = df[features].copy()
X_unscaled["Annual Income (k$)"] = X_unscaled["Annual Income (k$)"] * 1000

km_unscaled = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X_unscaled)
print("Ukuran kelompok TANPA penyamaan skala:", sorted(np.bincount(km_unscaled.labels_).tolist()))
print("Ukuran kelompok DENGAN penyamaan skala:", sorted(np.bincount(df["Cluster"]).tolist()))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
for j in range(5):
    members = df[df["Cluster"] == j]
    ax.scatter(members[features[0]], members[features[1]], s=22, color=palette[j % len(palette)],
               label=f"Kelompok {j}")
ax.scatter(centers_original[:, 0], centers_original[:, 1], s=200, marker="X",
           color="black", label="Centroid")
ax.set_xlabel(features[0])
ax.set_ylabel(features[1])
ax.set_title("K-Means (K = 5) pada Mall Customers")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Interpretasi

- Pada 8 titik, K-Means konvergen dalam 3 iterasi walau centroid awalnya buruk.
- Versi buatan sendiri menghasilkan centroid dan WCSS yang sama dengan scikit-learn.
- K = 4 sesuai data blobs. WCSS turun tajam sampai K = 4, lalu melandai.
- `k-means++` lebih stabil dan lebih cepat konvergen daripada `random` pada `n_init=1`.
- Tanpa penyamaan skala, pembagian hampir hanya mengikuti fitur berskala besar.

## Latihan

1. Ganti centroid awal 8 titik menjadi (1, 1) dan (1, 2). Berapa iterasi dan WCSS tiap iterasi?
2. Jalankan K-Means pada Mall Customers dengan K = 3, 4, 6. Bandingkan ukuran kelompok dan WCSS.
3. Hitung manual data 1D 1, 2, 6, 7, 8 (K = 2, centroid awal 1 dan 2), lalu cocokkan dengan `kmeans_scratch`.
4. Jalankan eksperimen `random` dengan `n_init=10` dan `n_init=3`.
5. Jalankan K = 8 pada 8 titik. Mengapa WCSS = 0 tidak berguna untuk memilih K?
6. Pelanggan baru (pendapatan 60, skor 50) masuk kelompok berapa?

Jawaban ada di halaman bab.

## Ringkasan

- K-Means: assignment lalu update, diulang sampai konvergen.
- WCSS tidak pernah naik, sehingga algoritma berhenti di local minimum.
- `k-means++` dan `n_init` mengurangi risiko hasil buruk.
- K dipilih dengan elbow dan pemahaman domain.
- Skalakan fitur sebelum K-Means.